In [11]:
import pandas as pd
from pathlib import Path
from natsort import natsorted
from src.csv_pages_to_names import read_files
from collections import defaultdict
from tqdm.notebook import tqdm
import re

## Metadata

In [12]:
def book_id_to_path(id, book_id_to_book, csv_data_directory):
    book_info = book_id_to_book[id]
    start_year = book_info["year_start"]
    end_year = book_info["year_end"]
    source = book_info["source"].lower()

    return Path(csv_data_directory).glob(
        f"*{book_info["parish_normalized"]}/*{start_year}-{end_year}_{source}"
    )

In [13]:
csv_data_directory = "data/migration-data-csv-release-v1"
book_ids_df = pd.read_json("outputs/json/book_ids.json", orient="index")
book_ids_df.columns = ["parish_normalized", "year_start", "year_end", "source"]
book_ids_df = book_ids_df.rename_axis("book_id")
book_ids_to_info = book_ids_df.to_dict(orient="index")

id_to_book = book_ids_df.to_dict(orient="index")
book_path_stems = tuple(tuple(i) for i in book_ids_df.index.map(
    lambda x: book_id_to_path(x, id_to_book, csv_data_directory)
).to_list())
book_path_stems = pd.DataFrame({"stem": tuple(i[0].stem if len(i) > 0 else None for i in book_path_stems)}, index=book_ids_df.index)

book_to_id = {
    v: k
    for k, v in (
        book_ids_df["parish_normalized"]
        + "_" + book_path_stems["stem"]
    )
    .to_dict()
    .items()
}

In [14]:
original_book_metadata = pd.read_csv("data/csv/Moving_record_parishes_with_formats_v2.csv")
original_book_metadata = original_book_metadata.dropna(subset="url")
original_book_metadata["book_id"] = original_book_metadata["url"].apply(lambda x: int(x.split("=")[-1]))
original_book_metadata = original_book_metadata.drop("url", axis="columns")

In [15]:
event_year_df = pd.read_csv("outputs/csv/year_predictions_extended.csv")
book_migration_metadata = pd.read_csv("outputs/csv/book_migration_direction_info.csv")
parish_df = pd.read_csv("data/csv/parish.csv")

In [16]:
print_type_columns = pd.read_csv("data/csv/Moving_record_formats_v2.csv")
print_type_columns["side"] = print_type_columns["sarakkeiden lkm"].apply(
    lambda x: re.sub(r"[\d\s]", "", x)
)
df_cols = list(print_type_columns.columns[6:])
df_cols.insert(0, df_cols.pop(-1))
df_cols.insert(0, "print type")
print_type_columns = print_type_columns[df_cols]

printed_cols = (
    print_type_columns[df_cols[2:]]
    .astype(str)
    .to_records(index=False)
    .tolist()
)
for i, p_type_cols in enumerate(printed_cols):
    corrected = list(filter(lambda x: not pd.isna(x), p_type_cols))
    printed_cols[i] = corrected

print(printed_cols)

print_type_columns["sarakkeet"] = printed_cols
print_type_columns = print_type_columns.drop(df_cols[3:], axis="columns")

[['kuukausi', 'päivä', 'nimi ja sääty', 'miespuoli', 'naispuoli', '(syntymä)päivä', '(syntymä)paikka', 'empty', 'leski, nainut, naimaton', 'elatuskeino', '(muuttokirjan)päivä', '(muuttokirjan)paikka', 'lehti kirkonkirjasta', 'muistutuksia'], ['kuukausi', 'päivä', 'muuttokirjan numero', 'nimi ja sääty', 'miespuoli', 'naispuoli', '(syntymä)päivä', '(syntymä)paikka', 'empty', 'leski, nainut, naimaton', 'elatuskeino', 'muuttopaikka', 'lehti kirkonkirjasta', 'muistutuksia'], ['juokseva numero', 'muuttokirja sisäänannettiin(kuukausi)', 'muuttokirja sisäänannettiin(päivä)', 'merkitty pääkirjaan sivulle', 'sisäänmuuttaneitten sääty, nimi ja ammatti', 'mistä seurakunnasta muutettiin', 'muuttokirjan päivämäärä', 'numero', 'miespuolta', 'vaimopuolta', 'ilmoitusvastaanottamisesta'], ['juokseva numero', 'muuttokirja sisäänannettiin(kuukausi)', 'muuttokirja sisäänannettiin(päivä)', 'merkitty pääkirjaan sivulle', 'sisäänmuuttaneitten sääty, nimi ja ammatti', 'mistä seurakunnasta muutettiin', 'muuttok

In [17]:
print_type_columns.to_csv("outputs/csv/print_type_columns.csv", index=False)

## Gather migrations

In [18]:
book_migration_metadata.loc[book_migration_metadata["side"] == "both", "side"] = ""
book_migration_metadata["path"] = book_migration_metadata["path"].fillna("")

In [19]:
def default_d():
    return defaultdict(dict)


all_file_paths = defaultdict(default_d)
for parish_id, b_id, p, side, in_out in tqdm(
    book_migration_metadata[
        ["parish_id", "book_id", "path", "side", "in/out"]
    ].to_records(index=False)
):
    if p == "":
        continue

    all_file_paths[int(parish_id)][int(b_id)][in_out] = natsorted(
        map(str, Path(p).glob(f"*{side}*.csv"))
    )

  0%|          | 0/3922 [00:00<?, ?it/s]

In [20]:
id_page_side_to_year = event_year_df[~(event_year_df["book_id"] == 0)]
id_page_side_to_year["key"] = (
    id_page_side_to_year["book_id"].astype(str)
    + "-" + id_page_side_to_year["page"].astype(str)
    + "-" + id_page_side_to_year["side"]
)
id_page_side_to_year = id_page_side_to_year.set_index("key")
id_page_side_to_year = id_page_side_to_year["year"].to_dict()

In [21]:
print_type_and_side_to_cols = print_type_columns.set_index(["print type", "side"])["sarakkeet"].to_dict()

In [22]:
id_to_print_type = book_migration_metadata[["book_id", "in/out", "print_type"]].set_index(["book_id", "in/out"])["print_type"].to_dict()

In [24]:
x = 0
migration_rows = []
for parish_id, books in tqdm(all_file_paths.items()):
    for b_id, directions in books.items():
        book_source = book_ids_to_info[b_id]["source"].lower()
        for in_out, file_paths in directions.items():
            pages, names = read_files(file_paths)
            print_type = id_to_print_type[(b_id, in_out)]

            for i, page in enumerate(pages):
                file_idx = i
                file_path = file_paths[i]
                split_path = file_path.split("_")
                page_n = int(split_path[-3])
                side = split_path[-2]
                year = id_page_side_to_year.get(str(b_id) + "-" + str(page_n) + "-" + side, 0)
                cols = print_type_and_side_to_cols.get((print_type, side), [])

                for row_idx, row in enumerate(page):

                    if year == 0:
                        x += 1

                    r = {
                        "row_data": row[0],
                        "row_columns": cols,
                        "year_prediction": int(year),
                        "file_path": file_path,
                        "parish_id": parish_id,
                        "book_id": b_id,
                        "print_type": print_type,
                        "source": book_source,
                        "file_row": row_idx,
                        "in/out": in_out,
                        "side": side,
                    }
                    migration_rows.append(
                        r
                    )
print(
    x,
    len(migration_rows)
)

  0%|          | 0/468 [00:00<?, ?it/s]

1140883 6720256


In [25]:
import json


with open("outputs/json/all_migration_events.jsonl", "a") as fp:
    batch = []
    i = 0

    for x in tqdm(migration_rows):
        if i < 100000:
            batch.append(json.dumps(x, ensure_ascii=False) + "\n")
            i += 1

        if i >= 100000:
            fp.writelines(batch)
            batch = []
            i = 0

    fp.writelines(batch)

  0%|          | 0/6720256 [00:00<?, ?it/s]